# Generalized Additive Models (GAMs) for Environmental Analysis

## Learning Objectives
By the end of this notebook, you will:
- Understand GAMs and their interpretability advantages
- Learn to model non-linear relationships with smooth functions
- Apply spatial smoothers for geographic patterns
- Implement GAMs for coastal/environmental data analysis
- Communicate results clearly to stakeholders

## Why Generalized Additive Models?

### Key Advantages:
1. **Highly Interpretable**: Each predictor's effect is clearly visualizable
2. **Non-linear Relationships**: Captures curves and patterns naturally
3. **Spatial Smoothers**: Can model geographic/spatial dependencies
4. **Stakeholder Communication**: Results are easy to explain
5. **Flexible**: Works with various response distributions

### Best Suited When:
- Clear interpretability is essential for stakeholders
- Relationships are suspected to be gradual and non-linear
- Need to understand individual predictor influences
- Working with spatial/geographic data
- Transparency is more important than maximum accuracy


In [ ]:
# Import essential libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Install required packages (uncomment if needed)
# !pip install pygam scipy

from pygam import LinearGAM, s, f, l
from pygam import LogisticGAM
from scipy.interpolate import UnivariateSpline
from scipy.spatial.distance import cdist

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print("GAM (PyGAM) ready for environmental modeling")

## 1. Understanding GAMs: The Mathematical Foundation

GAMs extend linear models by using smooth functions:

**Linear Model**: y = β₀ + β₁x₁ + β₂x₂ + ... + ε

**GAM**: y = β₀ + s₁(x₁) + s₂(x₂) + ... + ε

Where s₁, s₂, ... are smooth functions (splines) that can capture non-linear patterns.

In [ ]:
# Create synthetic coastal erosion dataset with known non-linear patterns
np.random.seed(42)
n_samples = 1500

# Generate spatial coordinates (latitude/longitude style)
lat = np.random.uniform(30, 45, n_samples)  # Degrees latitude
lon = np.random.uniform(-125, -70, n_samples)  # Degrees longitude (US coast)

# Generate environmental predictors with realistic relationships
data = {
    'latitude': lat,
    'longitude': lon,
    'elevation_m': np.random.uniform(0, 50, n_samples),
    'distance_to_coast_km': np.random.uniform(0, 25, n_samples),
    'sea_level_rise_mm_yr': np.random.uniform(1, 8, n_samples),
    'wave_energy_kj': np.random.uniform(10, 500, n_samples),
    'sediment_supply': np.random.uniform(0, 1, n_samples),
    'temperature_c': 15 + 0.5 * (45 - lat) + np.random.normal(0, 2, n_samples),  # Temperature-latitude relationship
    'precipitation_mm': 500 + 300 * np.sin((lon + 100) * np.pi / 50) + np.random.normal(0, 100, n_samples)
}

df = pd.DataFrame(data)

# Create realistic erosion rate with complex non-linear relationships
def calculate_erosion_rate(row):
    """Calculate coastal erosion with realistic non-linear patterns."""
    
    # Elevation effect (exponential decay - higher elevation = less erosion)
    elevation_effect = 5 * np.exp(-0.1 * row['elevation_m'])
    
    # Sea level rise effect (accelerating impact)
    slr_effect = 0.3 * row['sea_level_rise_mm_yr'] ** 1.5
    
    # Wave energy (logarithmic relationship)
    wave_effect = 2 * np.log(row['wave_energy_kj'] + 1)
    
    # Distance protection (inverse relationship with saturation)
    distance_protection = 3 / (1 + np.exp(0.3 * (row['distance_to_coast_km'] - 10)))
    
    # Sediment supply (protective effect)
    sediment_protection = -2 * row['sediment_supply']
    
    # Temperature effect (quadratic - optimal around 18°C)
    temp_effect = 0.01 * (row['temperature_c'] - 18) ** 2
    
    # Precipitation effect (U-shaped - both drought and excess are bad)
    precip_effect = 0.000005 * (row['precipitation_mm'] - 800) ** 2
    
    # Spatial effect (latitude-based climate gradient)
    spatial_effect = 0.1 * np.sin((row['latitude'] - 35) * np.pi / 10)
    
    # Combine effects
    erosion_rate = (
        elevation_effect + slr_effect + wave_effect + 
        distance_protection + sediment_protection + 
        temp_effect + precip_effect + spatial_effect
    )
    
    return max(0.1, erosion_rate)  # Minimum erosion rate

# Calculate erosion rates
df['erosion_rate_m_yr'] = df.apply(calculate_erosion_rate, axis=1)

# Add noise
df['erosion_rate_m_yr'] += np.random.normal(0, 0.2, n_samples)
df['erosion_rate_m_yr'] = np.maximum(0.01, df['erosion_rate_m_yr'])  # Ensure positive

print(f"Created coastal erosion dataset:")
print(f"- {len(df)} observations")
print(f"- Erosion rate range: {df['erosion_rate_m_yr'].min():.2f} to {df['erosion_rate_m_yr'].max():.2f} m/yr")
print(f"- Geographic range: {df['latitude'].min():.1f}°-{df['latitude'].max():.1f}°N, {df['longitude'].min():.1f}°-{df['longitude'].max():.1f}°W")

# Display sample data
df.head()

In [ ]:
# Visualize the relationships in our data
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.ravel()

# Plot each predictor against erosion rate
predictors = ['elevation_m', 'sea_level_rise_mm_yr', 'wave_energy_kj', 
             'distance_to_coast_km', 'sediment_supply', 'temperature_c',
             'precipitation_mm', 'latitude', 'longitude']

for i, pred in enumerate(predictors):
    axes[i].scatter(df[pred], df['erosion_rate_m_yr'], alpha=0.5, s=5)
    axes[i].set_xlabel(pred.replace('_', ' ').title())
    axes[i].set_ylabel('Erosion Rate (m/yr)')
    axes[i].set_title(f'Erosion vs {pred.replace("_", " ").title()}')
    
    # Add trend line
    z = np.polyfit(df[pred], df['erosion_rate_m_yr'], 1)
    p = np.poly1d(z)
    axes[i].plot(df[pred].sort_values(), p(df[pred].sort_values()), "r--", alpha=0.8)

plt.tight_layout()
plt.show()

# Geographic distribution
plt.figure(figsize=(12, 8))
scatter = plt.scatter(df['longitude'], df['latitude'], 
                     c=df['erosion_rate_m_yr'], 
                     cmap='RdYlBu_r', s=20, alpha=0.7)
plt.colorbar(scatter, label='Erosion Rate (m/yr)')
plt.xlabel('Longitude (°W)')
plt.ylabel('Latitude (°N)')
plt.title('Spatial Distribution of Coastal Erosion Rates')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Basic GAM Implementation

Let's start with a simple GAM to understand the smooth function concept:

In [ ]:
# Prepare data for GAM
X = df[['elevation_m', 'sea_level_rise_mm_yr', 'wave_energy_kj', 
        'distance_to_coast_km', 'sediment_supply', 'temperature_c']]
y = df['erosion_rate_m_yr']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create a basic GAM with smooth terms for all continuous variables
# s() creates a smooth spline function
# The numbers correspond to column indices in X
basic_gam = LinearGAM(
    s(0) +  # elevation_m
    s(1) +  # sea_level_rise_mm_yr  
    s(2) +  # wave_energy_kj
    s(3) +  # distance_to_coast_km
    s(4) +  # sediment_supply
    s(5),   # temperature_c
    n_splines=10  # Number of spline basis functions
)

# Fit the model
print("Fitting basic GAM...")
basic_gam.fit(X_train, y_train)

# Make predictions
y_pred = basic_gam.predict(X_test)

# Evaluate performance
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\nBasic GAM Performance:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² Score: {r2:.4f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.6, s=20)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Erosion Rate (m/yr)')
plt.ylabel('Predicted Erosion Rate (m/yr)')
plt.title(f'GAM: Predictions vs Actual (R² = {r2:.3f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Understanding Smooth Functions: The Heart of GAMs

The key advantage of GAMs is their interpretability through smooth function plots:

In [ ]:
# Plot the smooth functions (partial dependence plots)
feature_names = ['Elevation (m)', 'Sea Level Rise (mm/yr)', 'Wave Energy (kJ)', 
                'Distance to Coast (km)', 'Sediment Supply', 'Temperature (°C)']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, feature_name in enumerate(feature_names):
    # Generate partial dependence plot
    XX = basic_gam.generate_X_grid(term=i, n=100)
    
    # Get smooth function values
    pdep, confi = basic_gam.partial_dependence(term=i, X=XX, width=0.95)
    
    # Plot the smooth function
    axes[i].plot(XX[:, i], pdep, 'b-', linewidth=2, label='Smooth function')
    axes[i].fill_between(XX[:, i], confi[:, 0], confi[:, 1], alpha=0.3, color='blue', label='95% CI')
    
    # Add rug plot (data distribution)
    axes[i].plot(X_train.iloc[:, i], 
                [-0.1] * len(X_train), '|', color='red', alpha=0.5, markersize=1)
    
    axes[i].set_xlabel(feature_name)
    axes[i].set_ylabel('Partial Effect on Erosion Rate')
    axes[i].set_title(f'Smooth Function: {feature_name}')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()

plt.tight_layout()
plt.show()

# Print model summary
print("\nGAM Model Summary:")
print(basic_gam.summary())

## 4. Advanced GAM: Including Spatial Smoothers

For environmental data, spatial relationships are often crucial:

In [ ]:
# Include spatial coordinates for spatial smoothing
X_spatial = df[['elevation_m', 'sea_level_rise_mm_yr', 'wave_energy_kj', 
               'distance_to_coast_km', 'sediment_supply', 'temperature_c',
               'latitude', 'longitude']]
y_spatial = df['erosion_rate_m_yr']

# Split the data
X_train_sp, X_test_sp, y_train_sp, y_test_sp = train_test_split(
    X_spatial, y_spatial, test_size=0.2, random_state=42
)

# Create advanced GAM with spatial smoother
# te() creates a tensor product smooth (2D spatial smooth)
spatial_gam = LinearGAM(
    s(0) +  # elevation_m
    s(1) +  # sea_level_rise_mm_yr
    s(2) +  # wave_energy_kj
    s(3) +  # distance_to_coast_km
    s(4) +  # sediment_supply
    s(5) +  # temperature_c
    s(6, 7, n_splines=25),  # 2D spatial smooth (lat, lon)
    n_splines=15
)

# Fit the spatial model
print("Fitting spatial GAM...")
spatial_gam.fit(X_train_sp, y_train_sp)

# Make predictions
y_pred_sp = spatial_gam.predict(X_test_sp)

# Evaluate performance
mse_sp = mean_squared_error(y_test_sp, y_pred_sp)
rmse_sp = np.sqrt(mse_sp)
mae_sp = mean_absolute_error(y_test_sp, y_pred_sp)
r2_sp = r2_score(y_test_sp, y_pred_sp)

print(f"\nSpatial GAM Performance:")
print(f"RMSE: {rmse_sp:.4f} (vs {rmse:.4f} basic)")
print(f"MAE: {mae_sp:.4f} (vs {mae:.4f} basic)")
print(f"R² Score: {r2_sp:.4f} (vs {r2:.4f} basic)")
print(f"Improvement in R²: {r2_sp - r2:.4f}")

# Compare models
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Basic GAM
axes[0].scatter(y_test, y_pred, alpha=0.6, s=20, color='blue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Erosion Rate (m/yr)')
axes[0].set_ylabel('Predicted Erosion Rate (m/yr)')
axes[0].set_title(f'Basic GAM (R² = {r2:.3f})')
axes[0].grid(True, alpha=0.3)

# Spatial GAM
axes[1].scatter(y_test_sp, y_pred_sp, alpha=0.6, s=20, color='green')
axes[1].plot([y_test_sp.min(), y_test_sp.max()], [y_test_sp.min(), y_test_sp.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Erosion Rate (m/yr)')
axes[1].set_ylabel('Predicted Erosion Rate (m/yr)')
axes[1].set_title(f'Spatial GAM (R² = {r2_sp:.3f})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Visualizing Spatial Effects

One of GAMs' strongest features is the ability to visualize spatial patterns:

In [ ]:
# Create a grid for spatial visualization
lat_range = np.linspace(df['latitude'].min(), df['latitude'].max(), 50)
lon_range = np.linspace(df['longitude'].min(), df['longitude'].max(), 50)
lat_grid, lon_grid = np.meshgrid(lat_range, lon_range)

# Create prediction grid (fix other variables at their means)
grid_points = []
for lat, lon in zip(lat_grid.ravel(), lon_grid.ravel()):
    point = [
        X_spatial['elevation_m'].mean(),
        X_spatial['sea_level_rise_mm_yr'].mean(),
        X_spatial['wave_energy_kj'].mean(),
        X_spatial['distance_to_coast_km'].mean(),
        X_spatial['sediment_supply'].mean(),
        X_spatial['temperature_c'].mean(),
        lat,
        lon
    ]
    grid_points.append(point)

grid_points = np.array(grid_points)

# Get spatial smooth component
spatial_effect, _ = spatial_gam.partial_dependence(term=6, X=grid_points, width=0.95)
spatial_grid = spatial_effect.reshape(lat_grid.shape)

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Spatial smooth surface
contour = axes[0, 0].contourf(lon_grid, lat_grid, spatial_grid, levels=20, cmap='RdYlBu_r')
axes[0, 0].set_xlabel('Longitude (°W)')
axes[0, 0].set_ylabel('Latitude (°N)')
axes[0, 0].set_title('Spatial Effect on Erosion Rate')
plt.colorbar(contour, ax=axes[0, 0], label='Spatial Effect')

# 2. Observed vs predicted spatial pattern
scatter1 = axes[0, 1].scatter(df['longitude'], df['latitude'], 
                             c=df['erosion_rate_m_yr'], 
                             cmap='RdYlBu_r', s=15, alpha=0.7)
axes[0, 1].set_xlabel('Longitude (°W)')
axes[0, 1].set_ylabel('Latitude (°N)')
axes[0, 1].set_title('Observed Erosion Rates')
plt.colorbar(scatter1, ax=axes[0, 1], label='Erosion Rate (m/yr)')

# 3. Model predictions
all_pred = spatial_gam.predict(X_spatial)
scatter2 = axes[1, 0].scatter(df['longitude'], df['latitude'], 
                             c=all_pred, 
                             cmap='RdYlBu_r', s=15, alpha=0.7)
axes[1, 0].set_xlabel('Longitude (°W)')
axes[1, 0].set_ylabel('Latitude (°N)')
axes[1, 0].set_title('GAM Predicted Erosion Rates')
plt.colorbar(scatter2, ax=axes[1, 0], label='Predicted Rate (m/yr)')

# 4. Residuals
residuals = df['erosion_rate_m_yr'] - all_pred
scatter3 = axes[1, 1].scatter(df['longitude'], df['latitude'], 
                             c=residuals, 
                             cmap='RdBu', s=15, alpha=0.7)
axes[1, 1].set_xlabel('Longitude (°W)')
axes[1, 1].set_ylabel('Latitude (°N)')
axes[1, 1].set_title('Model Residuals')
plt.colorbar(scatter3, ax=axes[1, 1], label='Residual (m/yr)')

plt.tight_layout()
plt.show()

print(f"Spatial pattern analysis:")
print(f"- Mean absolute residual: {np.abs(residuals).mean():.3f} m/yr")
print(f"- Spatial pattern captured: {1 - (residuals.var() / df['erosion_rate_m_yr'].var()):.3%}")

## 6. Model Selection and Optimization

GAMs have hyperparameters that can be optimized:

In [ ]:
# Test different smoothing parameters
from pygam import LinearGAM
from sklearn.model_selection import cross_val_score

# Test different numbers of splines
spline_options = [5, 10, 15, 20, 25]
cv_scores = []

for n_splines in spline_options:
    gam = LinearGAM(
        s(0) + s(1) + s(2) + s(3) + s(4) + s(5),
        n_splines=n_splines
    )
    
    # Custom scoring function for GAM
    def gam_score(gam, X, y):
        gam.fit(X, y)
        return gam.score(X, y)
    
    # Manually implement cross-validation for GAM
    from sklearn.model_selection import KFold
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(X_train):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_val = X_train.iloc[val_idx]
        y_fold_train = y_train.iloc[train_idx]
        y_fold_val = y_train.iloc[val_idx]
        
        gam.fit(X_fold_train, y_fold_train)
        pred = gam.predict(X_fold_val)
        score = r2_score(y_fold_val, pred)
        scores.append(score)
    
    cv_scores.append(np.mean(scores))
    print(f"n_splines={n_splines}: CV R² = {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# Plot cross-validation results
plt.figure(figsize=(10, 6))
plt.plot(spline_options, cv_scores, 'bo-', linewidth=2, markersize=8)
plt.xlabel('Number of Splines')
plt.ylabel('Cross-Validation R² Score')
plt.title('GAM Model Selection: Number of Splines')
plt.grid(True, alpha=0.3)
plt.axvline(spline_options[np.argmax(cv_scores)], color='red', linestyle='--', 
           label=f'Optimal: {spline_options[np.argmax(cv_scores)]} splines')
plt.legend()
plt.tight_layout()
plt.show()

optimal_splines = spline_options[np.argmax(cv_scores)]
print(f"\nOptimal number of splines: {optimal_splines}")
print(f"Best CV score: {max(cv_scores):.4f}")

## 7. Feature Importance and Interpretation

GAMs provide excellent interpretability through statistical significance and smooth function analysis:

In [ ]:
# Create optimized GAM
optimal_gam = LinearGAM(
    s(0) + s(1) + s(2) + s(3) + s(4) + s(5),
    n_splines=optimal_splines
)
optimal_gam.fit(X_train, y_train)

# Get model statistics
print("GAM Model Statistics:")
print("=" * 40)
print(optimal_gam.summary())

# Calculate feature importance based on variance explained
feature_importance = []
feature_names = ['elevation_m', 'sea_level_rise_mm_yr', 'wave_energy_kj', 
                'distance_to_coast_km', 'sediment_supply', 'temperature_c']

for i, feature_name in enumerate(feature_names):
    # Get partial dependence
    XX = optimal_gam.generate_X_grid(term=i, n=100)
    pdep, _ = optimal_gam.partial_dependence(term=i, X=XX)
    
    # Calculate variance of the smooth function as importance measure
    importance = np.var(pdep)
    feature_importance.append(importance)

# Create feature importance DataFrame
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

# Normalize importance
importance_df['importance_normalized'] = importance_df['importance'] / importance_df['importance'].sum()

# Plot feature importance
plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(importance_df)), importance_df['importance_normalized'], alpha=0.7)
plt.yticks(range(len(importance_df)), importance_df['feature'])
plt.xlabel('Normalized Feature Importance (Variance of Smooth Function)')
plt.title('GAM Feature Importance Analysis')
plt.grid(True, alpha=0.3, axis='x')

# Add percentage labels
for i, bar in enumerate(bars):
    width = bar.get_width()
    plt.text(width + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{width:.1%}', ha='left', va='center')

plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
for i, (_, row) in enumerate(importance_df.iterrows()):
    print(f"{i+1}. {row['feature']}: {row['importance_normalized']:.2%}")

## 8. Communicating Results to Stakeholders

GAMs excel at creating interpretable visualizations for non-technical audiences:

In [ ]:
# Create stakeholder-friendly interpretation plots
fig = plt.figure(figsize=(20, 12))

# Create a 3x2 grid with larger plots for the most important features
gs = fig.add_gridspec(3, 4, height_ratios=[2, 2, 1], width_ratios=[1, 1, 1, 1])

# Top 4 most important features (larger plots)
top_features = importance_df.head(4)

for i, (_, row) in enumerate(top_features.iterrows()):
    feature_idx = feature_names.index(row['feature'])
    
    if i < 2:
        ax = fig.add_subplot(gs[0, i*2:(i+1)*2])  # Top row, spanning 2 columns
    else:
        ax = fig.add_subplot(gs[1, (i-2)*2:(i-1)*2])  # Second row, spanning 2 columns
    
    # Generate smooth function
    XX = optimal_gam.generate_X_grid(term=feature_idx, n=100)
    pdep, confi = optimal_gam.partial_dependence(term=feature_idx, X=XX, width=0.95)
    
    # Plot with enhanced styling
    ax.plot(XX[:, feature_idx], pdep, 'b-', linewidth=3, label='Effect on Erosion Rate')
    ax.fill_between(XX[:, feature_idx], confi[:, 0], confi[:, 1], 
                   alpha=0.3, color='blue', label='95% Confidence Interval')
    
    # Add data distribution
    ax2 = ax.twinx()
    ax2.hist(X_train.iloc[:, feature_idx], bins=30, alpha=0.3, color='gray', density=True)
    ax2.set_ylabel('Data Density', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    
    # Styling
    ax.set_xlabel(row['feature'].replace('_', ' ').title(), fontsize=12)
    ax.set_ylabel('Effect on Erosion Rate', fontsize=12, color='blue')
    ax.set_title(f'{row["feature"].replace("_", " ").title()}\n(Importance: {row["importance_normalized"]:.1%})', 
                fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='y', labelcolor='blue')

# Summary statistics in bottom row
ax_summary = fig.add_subplot(gs[2, :])
ax_summary.axis('off')

# Add text summary
summary_text = f"""
MODEL PERFORMANCE SUMMARY
• R² Score: {r2_score(y_test, optimal_gam.predict(X_test)):.3f} (explains {r2_score(y_test, optimal_gam.predict(X_test)):.1%} of erosion rate variance)
• RMSE: {np.sqrt(mean_squared_error(y_test, optimal_gam.predict(X_test))):.3f} m/yr
• Most influential factor: {importance_df.iloc[0]['feature'].replace('_', ' ').title()}
• Model Type: Generalized Additive Model (GAM) - Captures non-linear relationships while remaining interpretable

KEY INSIGHTS FOR COASTAL MANAGEMENT:
• Each curve shows how that factor alone influences erosion rates
• Steep curves indicate stronger effects; flat areas show minimal impact
• Confidence bands show uncertainty - wider bands mean less certain predictions
• Gray histograms show how much data we have for each range (more data = more reliable)
"""

ax_summary.text(0.02, 0.95, summary_text, transform=ax_summary.transAxes, 
               fontsize=11, verticalalignment='top', fontfamily='monospace',
               bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgray', alpha=0.7))

plt.suptitle('Coastal Erosion Risk Factors: GAM Analysis for Stakeholders', 
            fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.subplots_adjust(top=0.94)
plt.show()

# Create a simple interpretation table
print("\n" + "="*80)
print("SIMPLIFIED INTERPRETATION FOR STAKEHOLDERS")
print("="*80)

interpretations = {
    'elevation_m': 'Higher elevation provides strong protection against erosion',
    'sea_level_rise_mm_yr': 'Faster sea level rise dramatically increases erosion risk',
    'wave_energy_kj': 'Higher wave energy directly increases erosion rates',
    'distance_to_coast_km': 'Areas farther from coast experience lower erosion',
    'sediment_supply': 'More sediment helps protect against erosion',
    'temperature_c': 'Temperature effects on erosion are complex and non-linear'
}

for _, row in importance_df.iterrows():
    feature = row['feature']
    importance = row['importance_normalized']
    interpretation = interpretations.get(feature, 'No interpretation available')
    
    print(f"\n{feature.replace('_', ' ').title()}:")
    print(f"  Importance: {importance:.1%} of total model influence")
    print(f"  Effect: {interpretation}")

## 9. Uncertainty Quantification

GAMs provide natural uncertainty estimates through confidence intervals:

In [ ]:
# Prediction with confidence intervals
predictions, pred_intervals = optimal_gam.prediction_intervals(X_test, width=0.95)

# Calculate prediction uncertainty metrics
uncertainty = pred_intervals[:, 1] - pred_intervals[:, 0]
relative_uncertainty = uncertainty / predictions

# Analyze uncertainty patterns
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Predictions with confidence intervals
sorted_idx = np.argsort(predictions)
axes[0, 0].fill_between(range(len(predictions)), 
                       pred_intervals[sorted_idx, 0], 
                       pred_intervals[sorted_idx, 1], 
                       alpha=0.3, label='95% Confidence Interval')
axes[0, 0].plot(range(len(predictions)), predictions[sorted_idx], 'b-', label='Predictions')
axes[0, 0].scatter(range(len(predictions)), y_test.iloc[sorted_idx], 
                  alpha=0.6, s=10, color='red', label='Actual')
axes[0, 0].set_xlabel('Sorted Test Samples')
axes[0, 0].set_ylabel('Erosion Rate (m/yr)')
axes[0, 0].set_title('Predictions with Uncertainty')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Uncertainty vs prediction magnitude
axes[0, 1].scatter(predictions, uncertainty, alpha=0.6, s=15)
axes[0, 1].set_xlabel('Predicted Erosion Rate (m/yr)')
axes[0, 1].set_ylabel('Prediction Uncertainty (95% CI width)')
axes[0, 1].set_title('Uncertainty vs Prediction Magnitude')
axes[0, 1].grid(True, alpha=0.3)

# 3. Relative uncertainty distribution
axes[1, 0].hist(relative_uncertainty * 100, bins=30, alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Relative Uncertainty (%)')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Relative Uncertainty')
axes[1, 0].axvline(np.median(relative_uncertainty) * 100, color='red', linestyle='--',
                  label=f'Median: {np.median(relative_uncertainty)*100:.1f}%')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Coverage probability (should be ~95%)
coverage = ((y_test >= pred_intervals[:, 0]) & 
           (y_test <= pred_intervals[:, 1])).mean()

# Plot actual vs predicted with uncertainty coloring
scatter = axes[1, 1].scatter(y_test, predictions, c=uncertainty, 
                            cmap='viridis', s=20, alpha=0.7)
axes[1, 1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
               'r--', lw=2, label='Perfect Prediction')
axes[1, 1].set_xlabel('Actual Erosion Rate (m/yr)')
axes[1, 1].set_ylabel('Predicted Erosion Rate (m/yr)')
axes[1, 1].set_title(f'Predictions Colored by Uncertainty\nCoverage: {coverage:.1%}')
axes[1, 1].legend()
plt.colorbar(scatter, ax=axes[1, 1], label='Uncertainty (CI width)')

plt.tight_layout()
plt.show()

print(f"Uncertainty Analysis Results:")
print(f"- Coverage probability: {coverage:.3f} (should be ~0.95)")
print(f"- Mean uncertainty: {uncertainty.mean():.3f} m/yr")
print(f"- Median relative uncertainty: {np.median(relative_uncertainty):.1%}")
print(f"- 90% of predictions have uncertainty < {np.percentile(uncertainty, 90):.3f} m/yr")

## 10. Comparing GAMs with Other Methods

Let's compare GAM performance with other regression methods:

In [ ]:
# Compare GAM with other methods
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Prepare data for sklearn models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Polynomial (degree 2)': Pipeline([
        ('poly', PolynomialFeatures(degree=2, include_bias=False)),
        ('ridge', Ridge(alpha=10.0))
    ]),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'GAM (Optimal)': optimal_gam
}

# Train and evaluate all models
results = {}

for name, model in models.items():
    if name == 'GAM (Optimal)':
        # GAM already trained
        pred = model.predict(X_test)
    elif name in ['Linear Regression', 'Ridge Regression', 'Polynomial (degree 2)']:
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
    
    # Calculate metrics
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    
    results[name] = {
        'R²': r2,
        'RMSE': rmse,
        'MAE': mae,
        'predictions': pred
    }

# Create comparison DataFrame
comparison_df = pd.DataFrame(results).T[['R²', 'RMSE', 'MAE']]
comparison_df = comparison_df.round(4)

print("Model Comparison Results:")
print("=" * 50)
print(comparison_df)

# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, (name, result) in enumerate(results.items()):
    axes[i].scatter(y_test, result['predictions'], alpha=0.6, s=15)
    axes[i].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
                'r--', lw=2)
    axes[i].set_xlabel('Actual Erosion Rate (m/yr)')
    axes[i].set_ylabel('Predicted Erosion Rate (m/yr)')
    axes[i].set_title(f'{name}\nR² = {result["R²"]:.3f}, RMSE = {result["RMSE"]:.3f}')
    axes[i].grid(True, alpha=0.3)

# Hide the last subplot if there are only 5 models
if len(results) < 6:
    axes[5].set_visible(False)

plt.tight_layout()
plt.show()

# Bar plot of R² scores
plt.figure(figsize=(12, 6))
models_list = list(results.keys())
r2_scores = [results[model]['R²'] for model in models_list]

bars = plt.bar(models_list, r2_scores, alpha=0.7, color='skyblue', edgecolor='navy')

# Highlight GAM
gam_idx = models_list.index('GAM (Optimal)')
bars[gam_idx].set_color('orange')
bars[gam_idx].set_edgecolor('red')
bars[gam_idx].set_linewidth(2)

plt.xlabel('Model Type')
plt.ylabel('R² Score')
plt.title('Model Performance Comparison')
plt.xticks(rotation=45, ha='right')
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, score in zip(bars, r2_scores):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.005,
             f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Key Takeaways

### When to Use GAMs:

✅ **Perfect for:**
- Environmental monitoring with gradual, non-linear relationships
- Stakeholder communication requiring clear interpretability
- Spatial/geographic data with spatial dependencies
- Scientific studies needing transparent methodology
- Policy decisions requiring explainable predictions

### Key Advantages Demonstrated:
1. **Interpretability**: Every relationship is visualizable and explainable
2. **Non-linear Modeling**: Captures curves and thresholds naturally
3. **Uncertainty Quantification**: Provides confidence intervals
4. **Spatial Modeling**: Can include explicit spatial smoothers
5. **Stakeholder Communication**: Results are easily explained

### GAM vs Other Methods:
- **vs Linear Models**: Captures non-linearity while remaining interpretable
- **vs Random Forest**: More interpretable, better for smooth relationships
- **vs Neural Networks**: Much more interpretable, better uncertainty estimates
- **vs Gradient Boosting**: More interpretable, better for stakeholder communication

### Best Practices:
- Use cross-validation to select smoothing parameters
- Examine smooth functions for scientific plausibility
- Include spatial terms for geographic data
- Always provide confidence intervals
- Create stakeholder-friendly visualizations

GAMs are ideal when transparency and interpretability are as important as predictive performance!